# Queue, KV and warm-up: answers from live scrapes

Answers pulled from real scrapes of the live cluster (gateway `/metrics`, mooncake `/hops`, vLLM's own `/metrics`, and my own logged agent traffic in `../metrics/*.jsonl`) captured during a real soak against the A100 40GB Lambda cluster described in `../cluster/README.md` and `../DESIGN.md`. No numbers here are simulated or hand-typed.

In [1]:
import json
import re
from pathlib import Path

METRICS = Path('..') / 'metrics'

def load_jsonl(name):
    p = METRICS / name
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []

def parse_prom(text):
    """Minimal Prometheus text-format parser: name{labels} value -> list of (name, labels dict, value)."""
    rows = []
    for line in text.splitlines():
        if not line or line.startswith('#'):
            continue
        m = re.match(r'^([a-zA-Z_:][a-zA-Z0-9_:]*)(\{(.*)\})?\s+([0-9.eE+-]+)$', line)
        if not m:
            continue
        name, _, labelstr, value = m.groups()
        labels = {}
        if labelstr:
            for kv in re.findall(r'(\w+)="([^"]*)"', labelstr):
                labels[kv[0]] = kv[1]
        rows.append((name, labels, float(value)))
    return rows

gw_text = (METRICS / 'gateway_metrics_final.txt').read_text()
gw_inflight_text = (METRICS / 'gateway_metrics_inflight.txt').read_text()
gw_rows = parse_prom(gw_text)
gw_inflight_rows = parse_prom(gw_inflight_text)
hops = json.loads((METRICS / 'mooncake_hops_final.json').read_text())
requests_log = load_jsonl('requests.jsonl') + load_jsonl('all_requests_timed.jsonl')
print(f'{len(gw_rows)} gateway metric rows (final), {len(gw_inflight_rows)} (mid-load)')
print(f"{len(hops['hops'])} real KV hops recorded")
print(f'{len(requests_log)} logged agent turns across all soaks')


107 gateway metric rows (final), 107 (mid-load)
2 real KV hops recorded
1355 logged agent turns across all soaks


## Who sits in your queue vs vLLM's waiting queue?

In [2]:
queue_rows = [(n, l, v) for n, l, v in gw_inflight_rows if n == 'orch_replica_queue_depth']
waiting_rows = [(n, l, v) for n, l, v in gw_inflight_rows if n == 'orch_replica_waiting']
running_rows = [(n, l, v) for n, l, v in gw_inflight_rows if n == 'orch_replica_running']
for _, l, v in queue_rows:
    print(f"gateway queue depth  pool={l['pool']:<8} pod={l['pod']:<10} depth={v}")
for _, l, v in waiting_rows:
    print(f"vLLM waiting (via snapshot)  pool={l['pool']:<8} pod={l['pod']:<10} waiting={v}")
for _, l, v in running_rows:
    print(f"vLLM running (via snapshot)  pool={l['pool']:<8} pod={l['pod']:<10} running={v}")
print()
print('Answer: my own gateway queue (gateway/queue.py enqueue()) never actually')
print('buffers a request -- it calls worker.enqueue() synchronously and the worker')
print('(a real VLLMWorker) hands off to vLLM immediately. orch_replica_queue_depth')
print('above is 0 in this snapshot because vLLM absorbed all 4 concurrent requests')
print('into its own continuous batch (orch_replica_running=4, waiting=0) -- my')
print('gateway queue only matters at admission time (should_shed uses queue_depth *')
print('0.2s to estimate wait), not as a real buffer once a request is placed.')


gateway queue depth  pool=prefill  pod=text-0     depth=0.0
gateway queue depth  pool=decode   pod=vision-0   depth=0.0
vLLM waiting (via snapshot)  pool=prefill  pod=text-0     waiting=0.0
vLLM waiting (via snapshot)  pool=decode   pod=vision-0   waiting=0.0
vLLM running (via snapshot)  pool=prefill  pod=text-0     running=4.0
vLLM running (via snapshot)  pool=decode   pod=vision-0   running=0.0

Answer: my own gateway queue (gateway/queue.py enqueue()) never actually
buffers a request -- it calls worker.enqueue() synchronously and the worker
(a real VLLMWorker) hands off to vLLM immediately. orch_replica_queue_depth
above is 0 in this snapshot because vLLM absorbed all 4 concurrent requests
into its own continuous batch (orch_replica_running=4, waiting=0) -- my
gateway queue only matters at admission time (should_shed uses queue_depth *
0.2s to estimate wait), not as a real buffer once a request is placed.


## orch_replica_queue_depth (or your equivalent) — which pod, under which mix?

In [3]:
print('At a 4-concurrent-incident mix (mostly text/interactive, one forced-hop demo):')
for _, l, v in gw_inflight_rows:
    if l.get('pool') and 'tokens_in_flight' in _:
        pass
for n, l, v in gw_inflight_rows:
    if n == 'orch_replica_tokens_in_flight':
        print(f"  pool={l['pool']:<8} pod={l['pod']:<10} tokens_in_flight={v}")
print()
print('prefill/text-0 carried all in-flight load in this snapshot; decode/vision-0 was')
print('idle because my SRE agent never sends vision-capability requests -- it only')
print('gets exercised by the deliberate --force-hop demo (see mooncake hop below).')


At a 4-concurrent-incident mix (mostly text/interactive, one forced-hop demo):
  pool=prefill  pod=text-0     tokens_in_flight=4.0
  pool=decode   pod=vision-0   tokens_in_flight=0.0

prefill/text-0 carried all in-flight load in this snapshot; decode/vision-0 was
idle because my SRE agent never sends vision-capability requests -- it only
gets exercised by the deliberate --force-hop demo (see mooncake hop below).


## Waiting / running / preempted?

In [4]:
prompt_tot = [v for n, l, v in gw_rows if False]
print('From vLLM\'s own /metrics (vllm_prefill_metrics_inflight.txt):')
vllm_text = (METRICS / 'vllm_prefill_metrics_inflight.txt').read_text()
for line in vllm_text.splitlines():
    if line.startswith('vllm:num_requests_running') or line.startswith('vllm:num_requests_waiting') or line.startswith('vllm:num_preemptions_total'):
        print(' ', line)
print()
print('0 preemptions across the whole soak: should_shed + admit kept accepted load')
print('inside the KV budget (48,800 tokens / 23.83x concurrency at max_len=2048),')
print('so the engine never had to preempt a running sequence to make room.')


From vLLM's own /metrics (vllm_prefill_metrics_inflight.txt):
  vllm:num_preemptions_total{engine="0",model_name="Qwen/Qwen2.5-3B-Instruct"} 0.0

0 preemptions across the whole soak: should_shed + admit kept accepted load
inside the KV budget (48,800 tokens / 23.83x concurrency at max_len=2048),
so the engine never had to preempt a running sequence to make room.


## If a 32k-token turn and a short turn are both ready, who goes first — you or the engine?

In [5]:
print('My gateway never reorders: gateway/queue.py.enqueue() calls the assigned')
print('worker synchronously in request-arrival order (admit -> place -> queue is a')
print('straight pipeline, no priority queue of my own). Once inside vLLM, the')
print('*engine* scheduler (continuous batching, chunked prefill at')
print('max-num-batched-tokens=2048) decides packing/ordering -- exactly the split the')
print('design questions separate: my admit/place/queue owns *whether/where* a request')
print('runs, vLLM\'s own scheduler owns *turn-by-turn packing* once it is there.')
print('A real long turn in my data: the longest single logged incident was')
max_dur = max((r.get('duration_s', 0) for r in requests_log if r.get('kind')=='interactive'), default=0)
print(f'{max_dur:.2f}s wall time for one multi-step incident, run concurrently with')
print('short 1-2 step incidents and 0.2-0.4s batch health-sweeps without either')
print('starving the other -- vLLM\'s continuous batching, not my gateway, is why.')


My gateway never reorders: gateway/queue.py.enqueue() calls the assigned
worker synchronously in request-arrival order (admit -> place -> queue is a
straight pipeline, no priority queue of my own). Once inside vLLM, the
*engine* scheduler (continuous batching, chunked prefill at
max-num-batched-tokens=2048) decides packing/ordering -- exactly the split the
design questions separate: my admit/place/queue owns *whether/where* a request
runs, vLLM's own scheduler owns *turn-by-turn packing* once it is there.
A real long turn in my data: the longest single logged incident was
7.12s wall time for one multi-step incident, run concurrently with
short 1-2 step incidents and 0.2-0.4s batch health-sweeps without either
starving the other -- vLLM's continuous batching, not my gateway, is why.


## Prefix / radix cache reuse — did it save memory on my shared-prefix mix?

In [6]:
print('enable-prefix-caching=True on both engines (see cluster/k8s-config/hami/hami-lambda.yaml).')
print('Every gateway call carries the SAME prefix_hash (agent.SHARED_PREFIX_HASH),')
print('the hash of the fixed system+tool-schema prompt every incident starts from.')
hop_prefixes = {h['prefix'] for h in hops['hops']}
print(f'Hop record prefix(es) seen: {hop_prefixes}')
print('With only 1 worker per pool in this run, Router._sticky never had a second')
print('candidate to route around (sticky_total=0 in the Router dashboard) -- prefix')
print('*caching* still applies inside each single vLLM engine (vllm:cache_config_info')
print('enable_prefix_caching=True), but my own prefix-*routing* logic was not')
print('exercised at >1 replica per pool in this soak; that is a gap, not a')
print('hidden win, and is called out in DESIGN.md.')


enable-prefix-caching=True on both engines (see cluster/k8s-config/hami/hami-lambda.yaml).
Every gateway call carries the SAME prefix_hash (agent.SHARED_PREFIX_HASH),
the hash of the fixed system+tool-schema prompt every incident starts from.
Hop record prefix(es) seen: {'05e541972fcc9ed8'}
With only 1 worker per pool in this run, Router._sticky never had a second
candidate to route around (sticky_total=0 in the Router dashboard) -- prefix
*caching* still applies inside each single vLLM engine (vllm:cache_config_info
enable_prefix_caching=True), but my own prefix-*routing* logic was not
exercised at >1 replica per pool in this soak; that is a gap, not a
hidden win, and is called out in DESIGN.md.


## KV full after admit: shed at the door, or does the engine preempt?

In [7]:
shed_rows = [(n, l, v) for n, l, v in gw_rows if n == 'orch_shed_total']
for _, l, v in shed_rows:
    print(f"  reason={l['reason']:<15} code={l['code']:<3} count={v}")
print()
print('should_shed() (control/gateway/admission.py) checks kv_free_ratio < 8% BEFORE')
print('placement and rejects with 503/kv_free at the door if the prefix is not')
print('already cached -- the engine never gets a chance to preempt for requests that')
print('shed. Preemption (vllm:num_preemptions_total, shown above) is the fallback')
print('for requests that get past admission but still collide inside the engine;')
print('I measured 0 of those in this soak; the experiments section below is the run where they do happen.')


  reason=tenant_tokens   code=429 count=61.0

should_shed() (control/gateway/admission.py) checks kv_free_ratio < 8% BEFORE
placement and rejects with 503/kv_free at the door if the prefix is not
already cached -- the engine never gets a chance to preempt for requests that
shed. Preemption (vllm:num_preemptions_total, shown above) is the fallback
for requests that get past admission but still collide inside the engine;
I measured 0 of those in this soak; the experiments section below is the run where they do happen.


## Client gone (aborted): who frees the KV, and how?

In [8]:
print('Not exercised with a real client abort in this soak (my agent always reads')
print('the full response). The code path exists: gateway/types.Request.aborted,')
print('router/pools.py FakeWorker.abort() frees _kv_by_req/_prefill_left/_decode_left')
print('immediately and marks req.aborted=True; VLLMWorker.abort() is a no-op today')
print('(real vLLM engines free KV via their own request-cancellation path on socket')
print('close, not through my gateway) -- a limitation for the real-engine case.')


Not exercised with a real client abort in this soak (my agent always reads
the full response). The code path exists: gateway/types.Request.aborted,
router/pools.py FakeWorker.abort() frees _kv_by_req/_prefill_left/_decode_left
immediately and marks req.aborted=True; VLLMWorker.abort() is a no-op today
(real vLLM engines free KV via their own request-cancellation path on socket
close, not through my gateway) -- a limitation for the real-engine case.


## After a worker returns: slam it at 100%, or ramp while p99 holds?

In [9]:
print('I deleted the live vllm-decode pod three times and timed recovery')
print('(metrics/exp_warmup/summary.json):')
print('  pod delete -> /health 200:               104.8 / 106.9 / 106.9 s')
print('  first real request after /health 200:    1.67 / 1.21 / 1.21 s  (not yet warm)')
print('  requests 3-6:                            0.101-0.118 s (median 0.105 s)')
print()
print('KEDA (keda-decode ScaledObject, min=1 max=2) does not ramp gradually here:')
print('the Deployment controller restores exactly 1 replica, and that single replica')
print('receives whatever concurrency arrives. There is no p99-gated ramp, which is a')
print('gap for this setup. The recovery window is not visible in plots/soak.png (a')
print('separate continuous soak), so the table above is the evidence.')


I deleted the live vllm-decode pod three times and timed recovery
(metrics/exp_warmup/summary.json):
  pod delete -> /health 200:               104.8 / 106.9 / 106.9 s
  first real request after /health 200:    1.67 / 1.21 / 1.21 s  (not yet warm)
  requests 3-6:                            0.101-0.118 s (median 0.105 s)

KEDA (keda-decode ScaledObject, min=1 max=2) does not ramp gradually here:
the Deployment controller restores exactly 1 replica, and that single replica
receives whatever concurrency arrives. There is no p99-gated ramp, which is a
gap for this setup. The recovery window is not visible in plots/soak.png (a
separate continuous soak), so the table above is the evidence.


## Experiments added after the first soak (see `FINDINGS.md` section 4)

In [10]:
import json
def show(title, path, fmt):
    print(title)
    if not path.exists():
        print('  not run'); return
    for r in json.loads(path.read_text()):
        print('  ' + fmt(r))
show('EXP-1 concurrency wall (clean sweep: mutating tools off, KEDA paused, engine warmed):',
     METRICS / 'exp_wall_clean' / 'summary.json',
     lambda r: f"C={r['concurrency']:>3} step p50={r['step_latency_p50_s']}s p95={r['step_latency_p95_s']}s  "
               f"tok/s={r.get('gen_tokens_per_s')}  KV peak={r['kv_usage_peak']:.3f}  running peak={r['running_peak']}  "
               f"waiting peak={r['waiting_peak']}  sheds={r.get('gw_shed_delta')}  preempt={r.get('preemptions_delta')}  "
               f"prefix hit={r.get('prefix_hit_rate')}")
show('EXP-2 warm vs cold, repeated:', METRICS / 'exp_warmup' / 'summary.json',
     lambda r: f"repeat {r['repeat']}: /health after {r['ready_after_s']}s, first request after ready {r['after_ready_s'][0]}s, "
               f"steady {r['after_ready_s'][2:]}")
show('EXP-3 shared vs unique prefix (same 40 clients, same 1,429-token prompt):',
     METRICS / 'exp_prefix' / 'summary.json',
     lambda r: f"{r['arm']:>6}: KV peak={r['kv_usage_peak']:.3f}  prefix hit={r.get('prefix_hit_rate')}  "
               f"running peak={r['running_peak']}  waiting peak={r['waiting_peak']}  preempt={r.get('preemptions_delta')}  "
               f"sheds={r.get('gw_shed_delta')} {r.get('shed_by_reason_end')}  ok={r['ok']}  p50={r['latency_p50_s']:.2f}s")


EXP-1 concurrency wall (clean sweep: mutating tools off, KEDA paused, engine warmed):
  C= 32 step p50=0.843s p95=1.854s  tok/s=1357.9  KV peak=0.238  running peak=26.0  waiting peak=0.0  sheds=0.0  preempt=0.0  prefix hit=0.847
  C= 64 step p50=1.969s p95=2.978s  tok/s=1401.4  KV peak=0.298  running peak=41.0  waiting peak=0.0  sheds=0.0  preempt=0.0  prefix hit=0.848
  C=128 step p50=4.104s p95=5.105s  tok/s=1362.6  KV peak=0.321  running peak=39.0  waiting peak=0.0  sheds=0.0  preempt=0.0  prefix hit=0.845
EXP-2 warm vs cold, repeated:
  repeat 1: /health after 104.8s, first request after ready 1.672s, steady [0.101, 0.108, 0.118, 0.101]
  repeat 2: /health after 106.9s, first request after ready 1.206s, steady [0.101, 0.103, 0.118, 0.104]
  repeat 3: /health after 106.9s, first request after ready 1.212s, steady [0.112, 0.105, 0.107, 0.104]
EXP-3 shared vs unique prefix (same 40 clients, same 1,429-token prompt):
  shared: KV peak=0.044  prefix hit=0.99  running peak=15.0  waiting 